In [1]:
import scanpy as sc
import pandas as pd
import numpy as np

In [2]:
scdata = sc.read_h5ad('/REDACTED-path/HypoMap3D_singlecell_control.h5ad')
scdata

AnnData object with n_obs × n_vars = 896959 × 56939
    obs: 'CellID', 'Dataset', 'Sample', 'total_counts', 'n_genes', 'pct_mito', 'Author', 'Source', 'Data_source_link', 'GEO', 'Publication_Link', 'Suspension', 'Technology', 'Author_Sex_harmonized', 'sex_per_sample_inferred', 'Strain', 'Diet', 'Treatment', 'Age', 'Mean_age_in_weeks', 'IsControl', 'L1_ID', 'L2_ID', 'L3_ID', 'L4_ID', 'L5_ID', 'L1_descr', 'L2_descr', 'L3_descr', 'L4_descr', 'L5_descr', 'L1_abc_like', 'L2_abc_like', 'L3_abc_like', 'L4_abc_like', 'L5_abc_like'
    var: 'gene_ids', 'IsMouseTF', 'IsHumanTFOrtholog', 'IsSourceGene', 'IsTargetGene', 'IsHvgExcludedGene', 'IsAnnotationExcludedGene', 'IsPriorityGene', 'IsTopGene', 'IsHvg'
    obsm: 'X_embed', 'X_spatial_umap', 'X_umap'

In [3]:
sc.pp.normalize_total(scdata, target_sum=10_000)

In [4]:
LEVEL = 4

In [5]:
assert scdata.obs[f"L{LEVEL}_ID"].cat.codes.tolist() == scdata.obs[f"L{LEVEL}_descr"].cat.codes.tolist() == scdata.obs[f"L{LEVEL}_abc_like"].cat.codes.tolist(), "no alignment"

### Compute average expression per cell type (CP10K)

In [6]:
from scipy.sparse import csr_matrix

celltypes = scdata.obs[f"L{LEVEL}_ID"].cat.categories.to_numpy()
assignment_codes = scdata.obs[f"L{LEVEL}_ID"].cat.codes.to_numpy()

n_cells = len(assignment_codes)
n_celltypes = len(celltypes)

A = csr_matrix(
    (
        np.ones(n_cells),
        (
            assignment_codes,
            np.arange(n_cells)
        )
    ),
    shape=(n_celltypes, n_cells),
    dtype=bool
)

A

<Compressed Sparse Row sparse matrix of dtype 'bool'
	with 896959 stored elements and shape (716, 896959)>

In [7]:
# Sum expression of all cells assigned to each celltype
ctX = A @ scdata.X

# Compute no. of cells per celltype
n_cells_per_celltype = np.asarray(A.sum(axis=1))

# Compute average by dividing by no. of cell per cell type
ctX = ctX.multiply(1/n_cells_per_celltype).toarray().astype(np.float32)

### Compile cell type information into an AnnData object

In [8]:
ctdata = sc.AnnData(
    X = ctX,
    obs = pd.DataFrame(
        {
            'descr': scdata.obs[f"L{LEVEL}_descr"].cat.categories,
            'abc_like': scdata.obs[f"L{LEVEL}_abc_like"].cat.categories,
            'n_cells': n_cells_per_celltype.flatten()
        },
        index=scdata.obs[f"L{LEVEL}_ID"].cat.categories
    ),
    var = scdata.var.copy()
)

In [9]:
ctdata.layers['log1p'] = ctdata.X.copy()
sc.pp.log1p(
    ctdata,
    layer='log1p'
)

In [10]:
ctdata.obsm['X_embed'] = A @ scdata.obsm['X_embed'] / n_cells_per_celltype
ctdata.obsm['X_umap'] = A @ scdata.obsm['X_umap'] / n_cells_per_celltype

### Add percentage of cell expression to `.layers`

In [11]:
ctdata.layers['pct'] = (
    (
        A @ (scdata.X > 0).astype(np.int32)
    )
    .multiply(1/n_cells_per_celltype)
    .toarray()
    .astype(np.float32)
)

### Add marker information to `.layers`

In [12]:
df_markers = pd.read_csv(f'/REDACTED-path/HypoMap3D_cell_type_marker_genes_L{LEVEL}.csv')

df_sig_markers = df_markers[
    df_markers['padj'] < 0.001
]

df_vsAll_markers = df_sig_markers[df_sig_markers['comparison'] == 'OneVsAll']
df_vsSister_markers = df_sig_markers[df_sig_markers['comparison'] == 'OneVsSisters']

In [13]:
ctdata.layers['is_marker_vs_all'] = (
    df_vsAll_markers
    .pivot_table(
        'padj',
        index='celltype_id',
        columns='gene_name'
    )
    .reindex(
        index=ctdata.obs.index,
        columns=ctdata.var.index
    )
    .fillna(1)
    .astype(np.float32)
    .values
    < 0.001
)

In [14]:
ctdata.layers['is_marker_vs_sisters'] = (
    df_vsSister_markers
    .pivot_table(
        'padj',
        index='celltype_id',
        columns='gene_name'
    )
    .reindex(
        index=ctdata.obs.index,
        columns=ctdata.var.index
    )
    .fillna(1)
    .astype(np.float32)
    .values
    < 0.001
)

### Add regional predictions to `.obsm` and flags to `.obs`

In [15]:
df_region_relmass = pd.read_csv('/REDACTED-path/HypoMap3D-data/v0.9/9_objects/spatial/regionwise_relative_masses/regionwise_relative_masses_wo_fibers.csv', index_col=0)
df_ct_region_relmass = df_region_relmass.loc[ctdata.obs.index]

ctdata.obs['spatial_flag'] = df_ct_region_relmass['flag']

df_ct_region_relmass = df_ct_region_relmass.drop(columns=['celltype_descr', 'celltype_abc_like', 'flag'])
ctdata.obsm['df_region_relmass'] = df_ct_region_relmass

### Add useful metadata to `.obs`

In [16]:
# Parent Cluster
map_child_parent = scdata.obs.set_index(f'L{LEVEL}_ID')[f"L{LEVEL-1}_ID"].to_dict()
ctdata.obs['parent_id'] = [map_child_parent[x] for x in ctdata.obs.index]

# No. of children
ctdata.obs['n_children'] = scdata.obs.groupby(f"L{LEVEL}_ID", observed=True)[f"L{LEVEL+1}_ID"].nunique()

# Average read depth and total counts
ctdata.obs['avg_n_counts'] = (A @ scdata.obs['total_counts']) / ctdata.obs['n_cells']
ctdata.obs['avg_n_genes'] = (A @ scdata.obs['n_genes']) / ctdata.obs['n_cells']

# Sex summaries
is_female = scdata.obs['sex_per_sample_inferred'] == 'female'
is_male = scdata.obs['sex_per_sample_inferred'] == 'male'

ctdata.obs['n_female'] = A @ is_female.astype(np.int32)
ctdata.obs['n_male'] = A @ is_male.astype(np.int32)

ctdata.obs['pct_female'] = ctdata.obs['n_female'] / (ctdata.obs['n_female'] + ctdata.obs['n_male'])
ctdata.obs['pct_male'] = ctdata.obs['n_male'] / (ctdata.obs['n_female'] + ctdata.obs['n_male'])

# Source summaries

ctdata.obs["n_contrib_samples"] = (
    scdata.obs
    .groupby([f"L{LEVEL}_ID", "Sample"], observed=True)
    .size()
    .ge(10)
    .groupby(f"L{LEVEL}_ID", observed=True)
    .sum()
)
ctdata.obs['n_contrib_datasets'] = (
    scdata.obs
    .groupby([f"L{LEVEL}_ID", "Dataset"], observed=True)
    .size()
    .ge(10)
    .groupby(f"L{LEVEL}_ID", observed=True)
    .sum()
)


In [26]:
ctdata

AnnData object with n_obs × n_vars = 716 × 56939
    obs: 'descr', 'abc_like', 'n_cells', 'spatial_flag', 'parent_id', 'n_children', 'avg_n_counts', 'avg_n_genes', 'n_female', 'n_male', 'pct_female', 'pct_male', 'n_contrib_samples', 'n_contrib_datasets'
    var: 'gene_ids', 'IsMouseTF', 'IsHumanTFOrtholog', 'IsSourceGene', 'IsTargetGene', 'IsHvgExcludedGene', 'IsAnnotationExcludedGene', 'IsPriorityGene', 'IsTopGene', 'IsHvg'
    uns: 'log1p'
    obsm: 'X_embed', 'X_umap', 'df_region_relmass'
    layers: 'log1p', 'pct', 'is_marker_vs_all', 'is_marker_vs_sisters'